# Model Evaluation

Choosing the right evaluation metric is just as important as choosing the right model.
This notebook covers classification metrics, regression metrics, cross-validation
strategies, and practical visualization of model performance.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris, load_breast_cancer, make_regression
from sklearn.model_selection import train_test_split

np.random.seed(42)

---
## 1. Classification Metrics

### 1.1 The Confusion Matrix

For a **binary** classification problem, every prediction falls into one of
four categories:

|  | Predicted Positive | Predicted Negative |
|--|--------------------|-----------------|
| **Actually Positive** | True Positive (TP) | False Negative (FN) |
| **Actually Negative** | False Positive (FP) | True Negative (TN) |

All classification metrics are derived from these four counts.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, classification_report
)

# Load breast cancer dataset (binary classification)
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, test_size=0.25, random_state=42, stratify=cancer.target
)

# Fit a logistic regression model
log_reg = LogisticRegression(max_iter=10000, random_state=42)
log_reg.fit(X_train, y_train)
y_pred = log_reg.predict(X_test)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print("Confusion Matrix:")
print(cm)
print(f"\nTN={tn}, FP={fp}, FN={fn}, TP={tp}")

In [ ]:
# Visualize the confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw counts
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=cancer.target_names, yticklabels=cancer.target_names,
            ax=axes[0])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix (Counts)")

# Normalized (proportions)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="Blues",
            xticklabels=cancer.target_names, yticklabels=cancer.target_names,
            ax=axes[1])
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")
axes[1].set_title("Confusion Matrix (Normalized by Row)")

plt.tight_layout()
plt.show()

### 1.2 Accuracy, Precision, Recall, F1-Score

**Accuracy** -- fraction of all predictions that are correct:

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

**Precision** -- of all predicted positives, how many are truly positive?

$$\text{Precision} = \frac{TP}{TP + FP}$$

High precision matters when **false positives are costly** (e.g., spam filter marking legitimate email).

**Recall (Sensitivity)** -- of all actual positives, how many did the model find?

$$\text{Recall} = \frac{TP}{TP + FN}$$

High recall matters when **false negatives are costly** (e.g., missing a disease diagnosis).

**F1-Score** -- the harmonic mean of precision and recall, balancing both:

$$F_1 = 2 \cdot \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

The harmonic mean penalizes extreme imbalances: if either precision or recall is
very low, F1 will be low.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Verify with manual calculations
print("=== Classification Metrics ===")
print(f"Accuracy:  {accuracy:.4f}  (manual: {(tp+tn)/(tp+tn+fp+fn):.4f})")
print(f"Precision: {precision:.4f}  (manual: {tp/(tp+fp):.4f})")
print(f"Recall:    {recall:.4f}  (manual: {tp/(tp+fn):.4f})")
print(f"F1-Score:  {f1:.4f}  (manual: {2*(precision*recall)/(precision+recall):.4f})")
print(f"\nOf {tp+fp} positive predictions, {tp} correct and {fp} wrong.")
print(f"Of {tp+fn} actual positives, found {tp} and missed {fn}.")

In [ ]:
# The classification_report gives all metrics per class in one call
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

### 1.3 Precision-Recall Tradeoff

There is an inherent **tradeoff** between precision and recall:

- Increasing the threshold for classifying as positive: **higher precision, lower recall**
- Decreasing the threshold: **lower precision, higher recall**

The right balance depends on the application. In medical testing, we typically
prefer high recall (catch all sick patients), even at the cost of some false alarms.

---
## 2. When Accuracy is Misleading: Imbalanced Classes

Suppose 95% of transactions are legitimate and only 5% are fraudulent.
A model that **always predicts "legitimate"** achieves 95% accuracy -- but it
catches zero fraud. Accuracy alone is useless here.

In [ ]:
# Simulate an imbalanced dataset
np.random.seed(42)
n_samples = 1000
n_fraud = 50  # only 5%

y_true_imb = np.array([0] * (n_samples - n_fraud) + [1] * n_fraud)

# "Dumb" model: always predicts class 0 (legitimate)
y_pred_dumb = np.zeros(n_samples, dtype=int)

# A slightly better model that catches some fraud
y_pred_better = y_true_imb.copy()
fraud_indices = np.where(y_true_imb == 1)[0]
legit_indices = np.where(y_true_imb == 0)[0]
y_pred_better[fraud_indices[:10]] = 0   # miss 10 frauds
y_pred_better[legit_indices[:20]] = 1   # 20 false alarms

print("=== Dumb Model (always predicts legitimate) ===")
print(f"Accuracy:  {accuracy_score(y_true_imb, y_pred_dumb):.4f}")
print(f"Precision: {precision_score(y_true_imb, y_pred_dumb, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_true_imb, y_pred_dumb):.4f}")
print(f"F1-Score:  {f1_score(y_true_imb, y_pred_dumb):.4f}")

print("\n=== Better Model ===")
print(f"Accuracy:  {accuracy_score(y_true_imb, y_pred_better):.4f}")
print(f"Precision: {precision_score(y_true_imb, y_pred_better):.4f}")
print(f"Recall:    {recall_score(y_true_imb, y_pred_better):.4f}")
print(f"F1-Score:  {f1_score(y_true_imb, y_pred_better):.4f}")

In [ ]:
# Visualize the confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, y_p, title in zip(
    axes,
    [y_pred_dumb, y_pred_better],
    ["Dumb Model (acc=95%)", "Better Model"]
):
    cm_imb = confusion_matrix(y_true_imb, y_p)
    sns.heatmap(cm_imb, annot=True, fmt="d", cmap="Oranges",
                xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"],
                ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title)

plt.tight_layout()
plt.show()

print("Lesson: The dumb model has 95% accuracy but 0% recall -- zero fraud caught.")
print("The better model actually finds 80% of fraud. Precision, recall, and F1")
print("tell a much more complete story than accuracy alone.")

---
## 3. Regression Metrics

For regression tasks, we measure how far predictions are from actual values.

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **MAE** | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Average absolute error; same units as target; robust to outliers |
| **MSE** | $\frac{1}{n}\sum(y_i - \hat{y}_i)^2$ | Penalizes large errors more heavily (squared) |
| **RMSE** | $\sqrt{\text{MSE}}$ | Same units as target; most commonly reported |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Proportion of variance explained (1.0 = perfect, 0.0 = mean baseline) |

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Generate synthetic regression data
X_reg, y_reg = make_regression(
    n_samples=200, n_features=1, noise=20, random_state=42
)
y_reg = y_reg + 100  # offset for realism

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Fit and predict
lr = LinearRegression()
lr.fit(X_train_r, y_train_r)
y_pred_r = lr.predict(X_test_r)

# Compute all metrics
mae = mean_absolute_error(y_test_r, y_pred_r)
mse = mean_squared_error(y_test_r, y_pred_r)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_r, y_pred_r)

print("=== Regression Metrics ===")
print(f"MAE:  {mae:.4f}   (avg prediction off by {mae:.1f} units)")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}   (model explains {r2:.1%} of variance)")

In [ ]:
# Manual verification of R²
ss_res = np.sum((y_test_r - y_pred_r) ** 2)
ss_tot = np.sum((y_test_r - y_test_r.mean()) ** 2)
r2_manual = 1 - ss_res / ss_tot

print(f"R² (sklearn): {r2:.6f}")
print(f"R² (manual):  {r2_manual:.6f}")
print(f"\nSS_res = {ss_res:.2f}  (residual sum of squares)")
print(f"SS_tot = {ss_tot:.2f}  (total sum of squares)")

In [ ]:
# Visualize: actual vs predicted + residuals
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Actual vs predicted
axes[0].scatter(y_test_r, y_pred_r, alpha=0.6, edgecolors="k")
min_val = min(y_test_r.min(), y_pred_r.min())
max_val = max(y_test_r.max(), y_pred_r.max())
axes[0].plot([min_val, max_val], [min_val, max_val], "r--", linewidth=2,
             label="Perfect prediction")
axes[0].set_xlabel("Actual", fontsize=12)
axes[0].set_ylabel("Predicted", fontsize=12)
axes[0].set_title(f"Actual vs Predicted (R² = {r2:.3f})", fontsize=13)
axes[0].legend()

# Residuals
residuals = y_test_r - y_pred_r
axes[1].scatter(y_pred_r, residuals, alpha=0.6, edgecolors="k")
axes[1].axhline(y=0, color="r", linestyle="--", linewidth=2)
axes[1].set_xlabel("Predicted", fontsize=12)
axes[1].set_ylabel("Residual (Actual - Predicted)", fontsize=12)
axes[1].set_title("Residuals Plot", fontsize=13)

plt.tight_layout()
plt.show()

---
## 4. Train/Test Split vs Cross-Validation

| Approach | Pros | Cons |
|----------|------|------|
| Single train/test split | Fast, simple | High variance -- result depends on the specific split |
| k-Fold cross-validation | More robust estimate | Slower (trains k models) |

Cross-validation is generally preferred for **model selection and comparison**.
A final train/test split (or a held-out set) is used for the **final evaluation**.

In [ ]:
# Demonstrate variability of a single split
from sklearn.neighbors import KNeighborsClassifier

iris = load_iris()
accuracies = []

for seed in range(50):
    Xtr, Xte, ytr, yte = train_test_split(
        iris.data, iris.target, test_size=0.3, random_state=seed
    )
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(Xtr, ytr)
    accuracies.append(accuracy_score(yte, knn.predict(Xte)))

print(f"Accuracy across 50 different random splits:")
print(f"  Min:  {min(accuracies):.4f}")
print(f"  Max:  {max(accuracies):.4f}")
print(f"  Mean: {np.mean(accuracies):.4f}")
print(f"  Std:  {np.std(accuracies):.4f}")

plt.figure(figsize=(8, 4))
plt.hist(accuracies, bins=15, edgecolor="k", alpha=0.7)
plt.axvline(np.mean(accuracies), color="red", linestyle="--",
            label=f"Mean = {np.mean(accuracies):.3f}")
plt.xlabel("Accuracy", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.title("Variability of Single Train/Test Split (50 seeds)", fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()

---
## 5. k-Fold Cross-Validation

k-Fold CV splits data into **k** folds, trains k models (each using a different
fold as the test set), and averages the results.

```
5-Fold Cross-Validation on 100 samples:

Fold 1: [Test: 1-20]  [Train: 21-100]
Fold 2: [Train: 1-20] [Test: 21-40]  [Train: 41-100]
Fold 3: [Train: 1-40] [Test: 41-60]  [Train: 61-100]
Fold 4: [Train: 1-60] [Test: 61-80]  [Train: 81-100]
Fold 5: [Train: 1-80] [Test: 81-100]
```

Common choices: `k=5` or `k=10` are standard. `k=n` (Leave-One-Out) is expensive
but maximizes training data.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

knn = KNeighborsClassifier(n_neighbors=5)

# 5-fold and 10-fold cross-validation
cv_scores_5 = cross_val_score(knn, iris.data, iris.target, cv=5, scoring="accuracy")
cv_scores_10 = cross_val_score(knn, iris.data, iris.target, cv=10, scoring="accuracy")

print("5-Fold CV:")
print(f"  Scores: {cv_scores_5}")
print(f"  Mean:   {cv_scores_5.mean():.4f} +/- {cv_scores_5.std():.4f}")

print("\n10-Fold CV:")
print(f"  Scores: {cv_scores_10}")
print(f"  Mean:   {cv_scores_10.mean():.4f} +/- {cv_scores_10.std():.4f}")

In [ ]:
# Using KFold directly for more control
kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_scores = []
for fold_num, (train_idx, test_idx) in enumerate(kf.split(iris.data), 1):
    X_tr, X_te = iris.data[train_idx], iris.data[test_idx]
    y_tr, y_te = iris.target[train_idx], iris.target[test_idx]

    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_tr, y_tr)
    score = accuracy_score(y_te, knn.predict(X_te))
    fold_scores.append(score)
    print(f"Fold {fold_num}: train={len(train_idx)}, test={len(test_idx)}, "
          f"accuracy={score:.4f}")

print(f"\nMean: {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}")

---
## 6. Stratified k-Fold for Classification

Standard `KFold` does **not** guarantee that each fold has the same class
distribution as the full dataset. This can cause problems, especially with
imbalanced classes.

**StratifiedKFold** preserves the percentage of samples in each class across folds.

**Best practice**: For classification tasks, always use StratifiedKFold. When you
pass an integer to `cv=` in `cross_val_score`, sklearn automatically uses
StratifiedKFold for classifiers.

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Stratified 5-Fold: class distribution per fold")
print(f"{'Fold':<6} {'Train distribution':<25} {'Test distribution'}")
print("-" * 60)

for fold_num, (train_idx, test_idx) in enumerate(skf.split(iris.data, iris.target), 1):
    train_dist = np.bincount(iris.target[train_idx])
    test_dist = np.bincount(iris.target[test_idx])
    print(f"{fold_num:<6} {str(train_dist):<25} {test_dist}")

print(f"\nOriginal distribution: {np.bincount(iris.target)}")

In [ ]:
# Compare KFold vs StratifiedKFold on an imbalanced dataset
from sklearn.datasets import make_classification

X_imb, y_imb = make_classification(
    n_samples=200, n_features=10, n_classes=2,
    weights=[0.9, 0.1], random_state=42
)

print(f"Class distribution: {np.bincount(y_imb)}")
print(f"Class 1 is only {np.mean(y_imb):.1%} of the data\n")

kf_scores = cross_val_score(
    LogisticRegression(max_iter=1000), X_imb, y_imb,
    cv=KFold(n_splits=5, shuffle=True, random_state=42), scoring="f1"
)
skf_scores = cross_val_score(
    LogisticRegression(max_iter=1000), X_imb, y_imb,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), scoring="f1"
)

print(f"KFold F1:           {kf_scores.mean():.4f} +/- {kf_scores.std():.4f}")
print(f"StratifiedKFold F1: {skf_scores.mean():.4f} +/- {skf_scores.std():.4f}")
print("\nStratifiedKFold typically gives more stable and reliable estimates.")

---
## 7. The `scoring` Parameter in Scikit-Learn

The `scoring` parameter tells `cross_val_score` (and other tools) which metric
to optimize.

### Classification Scoring

| String | Metric |
|--------|--------|
| `"accuracy"` | Accuracy |
| `"precision"` / `"precision_macro"` | Precision (binary / multi-class) |
| `"recall"` / `"recall_macro"` | Recall |
| `"f1"` / `"f1_macro"` / `"f1_weighted"` | F1-score variants |
| `"roc_auc"` | Area Under the ROC Curve |

### Regression Scoring

| String | Metric |
|--------|--------|
| `"neg_mean_absolute_error"` | Negative MAE (negate to get MAE) |
| `"neg_mean_squared_error"` | Negative MSE |
| `"neg_root_mean_squared_error"` | Negative RMSE |
| `"r2"` | R-squared |

Sklearn uses the convention that **higher scores are better**, so loss metrics
like MSE are negated. To get the actual value, negate the returned score.

In [ ]:
# Compare multiple metrics simultaneously using cross_validate
from sklearn.model_selection import cross_validate

scoring_dict = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
}

knn = KNeighborsClassifier(n_neighbors=5)
cv_results = cross_validate(knn, iris.data, iris.target, cv=5, scoring=scoring_dict)

print("Multi-metric cross-validation on Iris (5-Fold):")
print("-" * 50)
for metric_name in scoring_dict:
    scores = cv_results[f"test_{metric_name}"]
    print(f"{metric_name:<20}: {scores.mean():.4f} +/- {scores.std():.4f}")

---
## 8. Visualization: Confusion Matrix with Seaborn Heatmap

A reusable function for plotting confusion matrices, applied to a multi-class problem.

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, title="Confusion Matrix",
                          normalize=False, figsize=(7, 5), cmap="Blues"):
    """
    Plot a confusion matrix as a seaborn heatmap.

    Parameters
    ----------
    y_true : array-like -- true labels
    y_pred : array-like -- predicted labels
    class_names : list of str -- names of each class
    title : str -- plot title
    normalize : bool -- if True, show proportions instead of counts
    figsize : tuple -- figure size
    cmap : str -- colormap
    """
    cm = confusion_matrix(y_true, y_pred)
    fmt = "d"
    if normalize:
        cm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        fmt = ".2%"

    plt.figure(figsize=figsize)
    sns.heatmap(cm, annot=True, fmt=fmt, cmap=cmap,
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, linecolor="gray")
    plt.xlabel("Predicted", fontsize=12)
    plt.ylabel("Actual", fontsize=12)
    plt.title(title, fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Multi-class example: Iris with KNN
X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=42, stratify=iris.target
)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_i, y_train_i)
y_pred_i = knn.predict(X_test_i)

plot_confusion_matrix(y_test_i, y_pred_i, iris.target_names,
                      title="KNN on Iris (Counts)")

plot_confusion_matrix(y_test_i, y_pred_i, iris.target_names,
                      title="KNN on Iris (Normalized)", normalize=True)

print(classification_report(y_test_i, y_pred_i, target_names=iris.target_names))

---
## 9. Putting It All Together: Model Comparison

Comparing multiple classifiers on the Iris dataset using cross-validation.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

models = {
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Logistic Regression": LogisticRegression(max_iter=10000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "SVM (RBF)": SVC(random_state=42),
}

results = {}
for name, model in models.items():
    scores = cross_val_score(model, iris.data, iris.target, cv=10, scoring="accuracy")
    results[name] = scores
    print(f"{name:<25}: {scores.mean():.4f} +/- {scores.std():.4f}")

In [ ]:
# Box plot comparison
results_df = pd.DataFrame(results)

plt.figure(figsize=(9, 5))
results_df.boxplot(grid=False, vert=True)
plt.ylabel("Accuracy (10-Fold CV)", fontsize=12)
plt.title("Model Comparison on Iris Dataset", fontsize=14)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

---
## 10. Key Takeaways

1. **Classification metrics**: Accuracy alone is often insufficient. Use precision,
   recall, F1-score, and the confusion matrix for a complete picture.

2. **Imbalanced classes** make accuracy misleading. Prefer F1, precision, or recall
   depending on whether false positives or false negatives are more costly.

3. **Regression metrics**: MAE is robust and interpretable, MSE/RMSE penalize large
   errors, and R-squared measures explained variance.

4. **Cross-validation** gives a more reliable performance estimate than a single
   train/test split. Use **StratifiedKFold** for classification.

5. The sklearn `scoring` parameter lets you specify exactly which metric to optimize.
   Use `cross_validate` to compute multiple metrics simultaneously.

6. Always **visualize** results: confusion matrices, actual-vs-predicted plots,
   and residual plots reveal patterns that numbers alone cannot.